# 分钟贡献与日内季节性

本单元面向已有分钟贡献的调用者，分别估计 RV、BPV 的多窗口每日季节曲线。每张曲线表的行是交易日，列是北京时间的分钟末端钟点。收益去季节可以选择任一种曲线；连接分钟数据、去季节及按采样区间汇总由 c02 完成。

## 输入与计算口径

估计输入是一个品种的一条连续主力分钟贡献表，包含 trading_date、bar_at 和指定贡献列。session_calendar_df 是该连续主力对应的单品种、1m Session 日历，包含 trading_date、session_start_at、session_end_at、is_night_session；调用者先用每日主力映射选择对应合约的日历。日历须覆盖完整研究范围及所需输出日，以识别 Session 的真实首次出现和夜盘阶段；不能随模型训练窗口移动而重新定义阶段起点。日历时间戳及 bar_at 使用 Asia/Shanghai 时区，交易日采用上游 trading_date。

价格准备可独立调用 compute_minute_contributions：使用 b02 的 adjusted_log_close，按 DataFrame 普通相邻行计算，不在午休、隔夜、Session 或换月处切断。已有分钟贡献时直接调用季节估计。

$$r_i=x_i-x_{i-1},\quad RV_i=r_i^2,\quad BPV_i=\frac{\pi}{2}|r_i||r_{i-1}|.$$

每日贡献先除以当日平均有效贡献：$u_{d,m}=N_d b_{d,m}/B_d$。缺失贡献不参与日总和和有效数；日总和非正时不产生日形状。先取固定窗口，再删除其中不可用值，不跨窗口借样本。每个分钟位置独立使用剩余有效历史。

跨日聚合使用原尺度 Huber 位置 M 估计，残差尺度固定为标准化 MAD，调节参数默认 1.345。全相同样本直接返回共同值；空样本、非恒定样本的零 MAD 或不收敛均记录明确原因。零因子保留。曲线最后按目标日日历中存在且估计有效的位置归一到均值 1；因子无效的位置保留缺失。

方法接口参考：[Huber 位置估计](https://www.statsmodels.org/stable/generated/statsmodels.robust.norms.estimate_location.html)、[标准化 MAD](https://www.statsmodels.org/stable/generated/statsmodels.robust.scale.mad.html)。

## 时段变化与初始期

日盘 Session 按起止钟点识别；新增 Session 独立初始化。夜盘的整体起止时段组合变化时，整个夜盘进入新阶段，不继承上一阶段历史。只有当日无夜盘时，保留之前的夜盘阶段，不把停夜盘日视为新制度。

正常曲线只使用目标交易日之前的数据。初始回填只作用于首次出现日位于训练范围内的 Session 阶段：从首次出现日起取同一段指定长度的未来样本，所有训练初始日都使用该段样本，历史完整后恢复正常滚动。估计样本及回填日期均截断于测试开始日之前；夜盘下一阶段的数据也不进入前一阶段的估计。

日期窗口采用左闭右开区间：正常窗口 $[D-L,D)$，初始窗口 $[S,\min(S+L,T))$。整数窗口表示交易日数，DateOffset 表示相应自然日期长度；缺失日期仍占用窗口长度。

定义写在 export 单元格；同名 Python 文件只加载定义。下面的合成示例显式执行，不读真实行情、不写 demo 数据、不启动实验或云任务。


In [1]:
from __future__ import annotations

from numbers import Integral
import numpy as np
import pandas as pd
from statsmodels.robust.norms import HuberT, estimate_location
from statsmodels.robust.scale import mad



In [2]:
SEASONALITY_FIELD_LABELS = {
    "minute_slot": "分钟末端钟点",
    "window_name": "季节窗口名",
    "session_phase": "Session 历史阶段",
    "history_start": "实际有效历史首日",
    "history_end": "实际有效历史末日",
    "sample_count": "有效历史样本数",
    "is_initial_backfill": "是否使用固定初始窗口",
    "huber_location": "归一化前的 Huber 位置估计",
    "status": "估计状态",
    "reason": "不可用原因",
}


In [3]:
def compute_minute_contributions(adjusted_main_minute_df: pd.DataFrame) -> pd.DataFrame:
    """从时间正序复权主力分钟表构造收益、RV 和 BPV；不重排或切断相邻行。"""
    if not adjusted_main_minute_df["bar_at"].is_monotonic_increasing:
        raise ValueError("贡献计算要求 bar_at 按真实时间正序排列")
    minute_contributions_df = adjusted_main_minute_df.copy()
    minute_contributions_df["log_return"] = minute_contributions_df["adjusted_log_close"].diff()
    minute_contributions_df["rv_contribution"] = minute_contributions_df["log_return"].pow(2)
    minute_contributions_df["bpv_contribution"] = (
        np.pi / 2 * minute_contributions_df["log_return"].abs()
        * minute_contributions_df["log_return"].shift(1).abs()
    )
    return minute_contributions_df


In [4]:
def _prepare_seasonality_inputs(minute_contributions_df, session_calendar_df, contribution_col):
    """共用日形状及 Session 阶段；仅检查本方法必需的连接和数学条件。"""
    calendar_df = session_calendar_df.loc[:, [
        "trading_date", "session_start_at", "session_end_at", "is_night_session",
    ]].copy()
    calendar_df["trading_date"] = pd.to_datetime(calendar_df["trading_date"]).dt.normalize()
    for column in ("session_start_at", "session_end_at"):
        calendar_df[column] = pd.to_datetime(calendar_df[column]).dt.tz_convert("Asia/Shanghai")
    calendar_df = calendar_df.sort_values(["trading_date", "session_start_at"])
    if calendar_df.empty:
        raise ValueError("Session 日历不能为空")
    trading_dates = pd.DatetimeIndex(calendar_df["trading_date"].unique(), name="trading_date")

    phase_starts = {}
    day_phase_ids = {}
    minute_slot_set = set()
    session_positions = []
    slot_cache = {}
    previous_night_signature = None
    night_phase_id = None
    for trading_date, day_calendar_df in calendar_df.groupby("trading_date", sort=False):
        night_df = day_calendar_df.loc[day_calendar_df["is_night_session"]]
        night_signature = tuple(
            (row.session_start_at.time(), row.session_end_at.time())
            for row in night_df.itertuples(index=False)
        )
        if night_signature and night_signature != previous_night_signature:
            night_phase_id = len(phase_starts)
            phase_starts[night_phase_id] = trading_date
            previous_night_signature = night_signature
        for row in day_calendar_df.itertuples(index=False):
            signature = (row.session_start_at.time(), row.session_end_at.time())
            if row.is_night_session:
                phase_id = night_phase_id
            else:
                if signature not in day_phase_ids:
                    day_phase_ids[signature] = len(phase_starts)
                    phase_starts[day_phase_ids[signature]] = trading_date
                phase_id = day_phase_ids[signature]
            if signature not in slot_cache:
                slot_cache[signature] = tuple(pd.date_range(
                    row.session_start_at + pd.Timedelta(minutes=1),
                    row.session_end_at, freq="min",
                ).time)
                if not slot_cache[signature]:
                    raise ValueError("Session 必须包含可用于本方法的一分钟位置")
            slots = slot_cache[signature]
            minute_slot_set.update(slots)
            session_positions.append((trading_date, phase_id, slots))
    minute_slots = pd.Index(sorted(minute_slot_set), name="minute_slot")
    phase_codes = np.full((len(trading_dates), len(minute_slots)), -1, dtype=np.int32)
    for trading_date, phase_id, slots in session_positions:
        day_position = trading_dates.get_loc(trading_date)
        slot_positions = minute_slots.get_indexer(slots)
        if (phase_codes[day_position, slot_positions] >= 0).any():
            raise ValueError("同一交易日的 Session 分钟钟点重叠，无法形成唯一每日曲线")
        phase_codes[day_position, slot_positions] = phase_id

    contribution_df = minute_contributions_df.loc[:, ["trading_date", "bar_at", contribution_col]].copy()
    contribution_df["trading_date"] = pd.to_datetime(contribution_df["trading_date"]).dt.normalize()
    contribution_df["minute_slot"] = pd.to_datetime(contribution_df["bar_at"]).dt.tz_convert(
        "Asia/Shanghai"
    ).dt.time
    contribution_values = contribution_df[contribution_col].to_numpy(dtype=float, na_value=np.nan)
    if (contribution_values[np.isfinite(contribution_values)] < 0).any():
        raise ValueError("RV/BPV 分钟贡献必须非负")
    contribution_df[contribution_col] = np.where(np.isfinite(contribution_values), contribution_values, np.nan)
    if contribution_df.duplicated(["trading_date", "minute_slot"]).any():
        raise ValueError("输入须为一条连续序列：交易日与分钟位置必须唯一")
    day_positions = trading_dates.get_indexer(contribution_df["trading_date"])
    slot_positions = minute_slots.get_indexer(contribution_df["minute_slot"])
    if (day_positions < 0).any() or (slot_positions < 0).any():
        raise ValueError("Session 日历未覆盖提供的分钟贡献")
    if (phase_codes[day_positions, slot_positions] < 0).any():
        raise ValueError("分钟贡献与目标 Session 位置无法连接")
    contribution_matrix_df = contribution_df.pivot(
        index="trading_date", columns="minute_slot", values=contribution_col,
    ).reindex(index=trading_dates, columns=minute_slots)
    daily_totals = contribution_matrix_df.sum(axis=1, min_count=1)
    valid_counts = contribution_matrix_df.count(axis=1)
    positive_totals = daily_totals.where(np.isfinite(daily_totals) & daily_totals.gt(0))
    daily_shapes_df = contribution_matrix_df.mul(valid_counts, axis=0).div(positive_totals, axis=0)
    return daily_shapes_df, phase_codes, phase_starts


def _huber_seasonal_location(samples, *, huber_t, huber_tol, huber_maxiter):
    """有效原尺度样本的固定 MAD Huber 位置；退化及数值失败具有明确状态。"""
    if not len(samples):
        return np.nan, "no_valid_history", "窗口内没有有效历史样本"
    if np.all(samples == samples[0]):
        return float(samples[0]), "ok", ""
    scale = float(mad(samples))
    if not np.isfinite(scale) or scale <= 0:
        return np.nan, "zero_scale", "非恒定样本的标准化 MAD 为零或无效"
    try:
        with np.errstate(divide="raise", invalid="raise", over="raise"):
            location = float(estimate_location(
                samples, scale, norm=HuberT(t=huber_t),
                maxiter=huber_maxiter, tol=huber_tol,
            ))
    except ValueError as error:
        return np.nan, "not_converged", str(error)
    except (FloatingPointError, OverflowError) as error:
        return np.nan, "numerical_failure", str(error)
    if not np.isfinite(location) or location < 0:
        return np.nan, "invalid_estimate", "Huber 位置估计非有限或为负"
    return location, "ok", ""


def _normalize_seasonality_curve(raw_curves_df, diagnostics_df, expected_positions):
    """只在目标日存在且有效的位置归一化；共用于滚动和回填结果。"""
    raw_values = raw_curves_df.to_numpy(copy=True)
    raw_values[~expected_positions] = np.nan
    valid_counts = np.isfinite(raw_values).sum(axis=1)
    curve_means = np.divide(
        np.nansum(raw_values, axis=1), valid_counts,
        out=np.full(len(raw_values), np.nan), where=valid_counts > 0,
    )
    valid_means = np.isfinite(curve_means) & (curve_means > 0)
    normalized_values = np.divide(
        raw_values, curve_means[:, None], out=np.full_like(raw_values, np.nan),
        where=valid_means[:, None],
    )
    invalid_dates = raw_curves_df.index[~valid_means]
    invalid_estimates = diagnostics_df["trading_date"].isin(invalid_dates) & diagnostics_df["status"].eq("ok")
    diagnostics_df.loc[invalid_estimates, "status"] = "invalid_curve_mean"
    diagnostics_df.loc[invalid_estimates, "reason"] = "每日曲线的有效均值非正或无效"
    return pd.DataFrame(normalized_values, index=raw_curves_df.index, columns=raw_curves_df.columns), diagnostics_df


## 函数调用与结果

estimate_intraday_seasonality 一次处理一种贡献、多种窗口；output_dates 默认取全部输入日历交易日，也可只输出指定日期。当日只有日历、尚无贡献时仍可生成开盘前曲线。返回普通字典：curves 按窗口名取每日矩阵，diagnostics 是按窗口／交易日／分钟位置记录的长表，parameters 保存贡献列及数学设置。

diagnostics 中的 huber_location 是均值归一化之前的成分，供初始回填重新组合每日曲线；最终使用 curves。history_start、history_end 是实际有效样本的首末日，sample_count 是固定窗口内去除无效值后剩余样本数。失败位置在曲线中保持缺失，原因写入 status、reason；其他位置继续估计。

backfill_initial_seasonality 沿用正常结果的窗口与 Huber 参数。它覆盖训练初始期的对应 Session 位置，即使正常结果已有少量历史估计；回填后重新归一化每日曲线。原结果及输入均不修改。该函数只在同一输入、日历与参数下使用，生成身份核对由调用方负责。

大实验先生成全局正常曲线，再按每次训练边界取得回填版本；历史各日使用各自每日曲线，测试日使用正常因果曲线。正式训练按整体训练边界取得回填版本后构造既定采样序列，逐点模型重估不重复估计季节曲线。


In [5]:
def estimate_intraday_seasonality(
    minute_contributions_df: pd.DataFrame,
    session_calendar_df: pd.DataFrame,
    *,
    contribution_col: str,
    lookbacks: dict,
    output_dates=None,
    huber_t: float = 1.345,
    huber_tol: float = 1e-6,
    huber_maxiter: int = 30,
) -> dict:
    """按交易日滚动估计一种贡献的多窗口曲线；此操作不使用未来贡献。

    lookbacks 为窗口名到正整数交易日数或正向 pd.DateOffset 的映射。
    output_dates 默认为全部日历交易日，可指定历史日或只有日历的下一日。
    返回 curves、diagnostics、parameters；曲线按窗口名选择。
    """
    if not isinstance(lookbacks, dict) or not lookbacks:
        raise ValueError("lookbacks 须为非空的窗口名到长度的映射")
    if not np.isfinite(huber_t) or huber_t <= 0 or not np.isfinite(huber_tol) or huber_tol <= 0:
        raise ValueError("Huber 阈值和容差须为有限正数")
    if isinstance(huber_maxiter, bool) or not isinstance(huber_maxiter, Integral) or huber_maxiter < 1:
        raise ValueError("huber_maxiter 须为正整数")
    daily_shapes_df, phase_codes, _ = _prepare_seasonality_inputs(
        minute_contributions_df, session_calendar_df, contribution_col,
    )
    trading_dates, minute_slots = daily_shapes_df.index, daily_shapes_df.columns
    for window_name, lookback in lookbacks.items():
        if not isinstance(window_name, str) or not window_name:
            raise ValueError("窗口名须为非空字符串")
        if isinstance(lookback, Integral) and not isinstance(lookback, bool):
            if lookback > 0:
                continue
        elif type(lookback) is pd.DateOffset:
            if all(day - lookback < day < day + lookback for day in (trading_dates[0], trading_dates[-1])):
                continue
        raise ValueError("窗口长度须为正整数交易日数或正向 pd.DateOffset")
    target_dates = trading_dates if output_dates is None else pd.DatetimeIndex(
        pd.to_datetime(output_dates), name="trading_date",
    ).normalize().unique().sort_values()
    target_positions = trading_dates.get_indexer(target_dates)
    if (target_positions < 0).any():
        raise ValueError("日历须覆盖全部输出交易日")

    shape_values = daily_shapes_df.to_numpy()
    curves_by_window = {}
    diagnostic_tables = []
    for window_name, lookback in lookbacks.items():
        raw_values = np.full((len(target_dates), len(minute_slots)), np.nan)
        records = []
        for output_position, day_position in enumerate(target_positions):
            day = trading_dates[day_position]
            history_start_position = max(0, day_position - int(lookback)) if isinstance(lookback, Integral) else int(
                trading_dates.searchsorted(day - lookback)
            )
            history_positions = np.arange(history_start_position, day_position)
            for slot_position in np.flatnonzero(phase_codes[day_position] >= 0):
                phase_id = int(phase_codes[day_position, slot_position])
                valid = (phase_codes[history_positions, slot_position] == phase_id) & np.isfinite(
                    shape_values[history_positions, slot_position]
                )
                sample_positions = history_positions[valid]
                location, status, reason = _huber_seasonal_location(
                    shape_values[sample_positions, slot_position],
                    huber_t=huber_t, huber_tol=huber_tol, huber_maxiter=huber_maxiter,
                )
                raw_values[output_position, slot_position] = location
                records.append((
                    window_name, day, minute_slots[slot_position], phase_id,
                    trading_dates[sample_positions[0]] if len(sample_positions) else pd.NaT,
                    trading_dates[sample_positions[-1]] if len(sample_positions) else pd.NaT,
                    len(sample_positions), False, location, status, reason,
                ))
        diagnostic_df = pd.DataFrame(records, columns=[
            "window_name", "trading_date", "minute_slot", "session_phase",
            "history_start", "history_end", "sample_count", "is_initial_backfill",
            "huber_location", "status", "reason",
        ])
        raw_curves_df = pd.DataFrame(raw_values, index=target_dates, columns=minute_slots)
        curves_by_window[window_name], diagnostic_df = _normalize_seasonality_curve(
            raw_curves_df, diagnostic_df, phase_codes[target_positions] >= 0,
        )
        diagnostic_tables.append(diagnostic_df)
    return {
        "curves": curves_by_window,
        "diagnostics": pd.concat(diagnostic_tables, ignore_index=True),
        "parameters": {
            "contribution_col": contribution_col, "lookbacks": dict(lookbacks),
            "huber_t": huber_t, "huber_tol": huber_tol, "huber_maxiter": int(huber_maxiter),
        },
    }


In [6]:
def backfill_initial_seasonality(
    seasonality_result: dict,
    minute_contributions_df: pd.DataFrame,
    session_calendar_df: pd.DataFrame,
    *,
    train_start,
    test_start,
) -> dict:
    """固定初始窗口覆盖训练初始期；返回副本，测试曲线保持正常因果结果。

    传入同一次正常估计所用的贡献与日历；窗口及 Huber 设置沿用 parameters。
    训练区间为 [train_start, test_start)，不在滚动模型训练首日重置阶段。
    """
    if "training_period" in seasonality_result:
        raise ValueError("回填输入须为 estimate_intraday_seasonality 返回的正常因果结果")
    train_start, test_start = pd.Timestamp(train_start).normalize(), pd.Timestamp(test_start).normalize()
    if pd.isna(train_start) or pd.isna(test_start) or train_start >= test_start:
        raise ValueError("训练边界须满足 train_start < test_start")
    parameters = seasonality_result["parameters"]
    daily_shapes_df, phase_codes, phase_starts = _prepare_seasonality_inputs(
        minute_contributions_df, session_calendar_df, parameters["contribution_col"],
    )
    trading_dates, minute_slots = daily_shapes_df.index, daily_shapes_df.columns
    shape_values = daily_shapes_df.to_numpy()
    curves_by_window = {}
    diagnostic_tables = []
    for window_name, lookback in parameters["lookbacks"].items():
        base_curves_df = seasonality_result["curves"][window_name]
        target_positions = trading_dates.get_indexer(base_curves_df.index)
        if (target_positions < 0).any() or not base_curves_df.columns.equals(minute_slots):
            raise ValueError("回填须使用与正常估计相同的日历位置")
        diagnostic_df = seasonality_result["diagnostics"].loc[
            seasonality_result["diagnostics"]["window_name"].eq(window_name)
        ].copy()
        raw_curves_df = diagnostic_df.pivot(
            index="trading_date", columns="minute_slot", values="huber_location",
        ).reindex(index=base_curves_df.index, columns=minute_slots)
        raw_values = raw_curves_df.to_numpy(copy=True)
        changed_dates = set()
        for phase_id, phase_start in phase_starts.items():
            if not train_start <= phase_start < test_start:
                continue
            start_position = trading_dates.get_loc(phase_start)
            if isinstance(lookback, Integral):
                initial_end_position = min(len(trading_dates), start_position + int(lookback))
                initial_history_positions = np.arange(start_position, initial_end_position)
                is_initial_target = (target_positions >= start_position) & (target_positions < start_position + int(lookback))
            else:
                initial_end = phase_start + lookback
                initial_history_positions = np.arange(start_position, trading_dates.searchsorted(initial_end))
                is_initial_target = (base_curves_df.index >= phase_start) & (base_curves_df.index < initial_end)
            initial_history_positions = initial_history_positions[
                trading_dates[initial_history_positions] < test_start
            ]
            initial_targets = np.flatnonzero(
                is_initial_target & (base_curves_df.index >= train_start) & (base_curves_df.index < test_start)
                & (phase_codes[target_positions] == phase_id).any(axis=1)
            )
            if not len(initial_targets):
                continue
            initial_slot_positions = np.flatnonzero((phase_codes[target_positions[initial_targets]] == phase_id).any(axis=0))
            for slot_position in initial_slot_positions:
                valid = (phase_codes[initial_history_positions, slot_position] == phase_id) & np.isfinite(
                    shape_values[initial_history_positions, slot_position]
                )
                sample_positions = initial_history_positions[valid]
                location, status, reason = _huber_seasonal_location(
                    shape_values[sample_positions, slot_position],
                    huber_t=parameters["huber_t"], huber_tol=parameters["huber_tol"],
                    huber_maxiter=parameters["huber_maxiter"],
                )
                affected_outputs = initial_targets[
                    phase_codes[target_positions[initial_targets], slot_position] == phase_id
                ]
                affected_dates = base_curves_df.index[affected_outputs]
                raw_values[affected_outputs, slot_position] = location
                changed_dates.update(affected_dates)
                affected_diagnostics = diagnostic_df["trading_date"].isin(affected_dates) & diagnostic_df["minute_slot"].eq(
                    minute_slots[slot_position]
                )
                diagnostic_df.loc[affected_diagnostics, [
                    "history_start", "history_end", "sample_count", "is_initial_backfill",
                    "huber_location", "status", "reason",
                ]] = [
                    trading_dates[sample_positions[0]] if len(sample_positions) else pd.NaT,
                    trading_dates[sample_positions[-1]] if len(sample_positions) else pd.NaT,
                    len(sample_positions), True, location, status, reason,
                ]
            # 原来无法归一化的其他有效位置也随整日曲线重新判定。
        if changed_dates:
            restore_ok = diagnostic_df["trading_date"].isin(changed_dates) & diagnostic_df["status"].eq("invalid_curve_mean")
            diagnostic_df.loc[restore_ok, "status"] = "ok"
            diagnostic_df.loc[restore_ok, "reason"] = ""
            normalized_df, diagnostic_df = _normalize_seasonality_curve(
                pd.DataFrame(raw_values, index=base_curves_df.index, columns=minute_slots),
                diagnostic_df, phase_codes[target_positions] >= 0,
            )
            curves_df = base_curves_df.copy()
            curves_df.loc[list(changed_dates)] = normalized_df.loc[list(changed_dates)]
        else:
            curves_df = base_curves_df.copy()
        curves_by_window[window_name] = curves_df
        diagnostic_tables.append(diagnostic_df)
    return {
        "curves": curves_by_window,
        "diagnostics": pd.concat(diagnostic_tables, ignore_index=True),
        "parameters": {**parameters, "lookbacks": dict(parameters["lookbacks"])},
        "training_period": {"train_start": train_start, "test_start": test_start},
    }


In [7]:
from pathlib import Path
import sys
from IPython.display import display

project_markers = [".git", ".env", "config/settings.py"]
current_path = Path.cwd().resolve()
for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        sys.path.insert(0, str(candidate_root / "04_Research"))
        break
else:
    raise RuntimeError("未找到项目根目录")
from config.settings import settings
from config.data_contracts import FUTURES_MINUTE_SCHEMA

# 本地仅做真实小范围内存试算，不写 demo 数据文件。
demo_variety = "RB"
demo_source_range = "train20100101-20241231_test20250101-20260930"
demo_start_date = pd.Timestamp("2024-12-31")
demo_end_date = pd.Timestamp("2025-01-02")
demo_test_start = pd.Timestamp("2025-01-01")
demo_sampling = "N15_gk_money_all_contracts_log_interpolate_fixed"
demo_base = candidate_root / "04_Research/a01_Methods"
demo_modeling = demo_base / "b04_ReturnVolatilityModeling"
demo_window_names = ("1y", "3y")
demo_history_points = 30  # 小范围内存试算；完整云 demo 固定使用 500 点。
# 完整 demo 由显式云入口执行；此范围不改变正式大实验截至 2026-06-30 的 scope。
demo_cloud_scope = {"start": "2010-01-01", "end": "2026-09-30", "test_start": "2025-01-01"}


In [8]:
from pathlib import Path
from datetime import datetime, timezone
import base64
import hashlib
import json
import os
import platform
import sys
import pyarrow as pa
import pyarrow.parquet as pq
from a01_Methods.b02_MainContinuousAdjustment.c01_MainContinuousAdjustment import arrow_content_sha256


def _real_demo_identity(root, producer, upstream, parameters, scope):
    """这四个单元的实际计算定义、调用配置和上游身份；不保存模型状态。"""
    root, producer = Path(root), Path(producer)
    names = ["c01_IntradaySeasonality"]
    if producer.stem != names[0]:
        names.append("c02_ModelObservations")
    if producer.stem in ("c03_ReturnModels", "c04_VolatilityModels"):
        names.append("c03_ReturnModels")
    if producer.stem == "c04_VolatilityModels":
        names.append(producer.stem)
    source_paths = {name: producer.parent / (name + ".ipynb") for name in names}
    if producer.stem != "c01_IntradaySeasonality":
        source_paths["sampling_positions"] = producer.parent.parent / "b03_Sampling/c01_SamplingPositions.ipynb"
    sources = {}
    for name, source_path in source_paths.items():
        notebook = json.loads(source_path.read_text(encoding="utf8"))
        source_tag = "export" if name == "sampling_positions" else "demo-dependency"
        source = "\n\n".join("".join(c["source"]) for c in notebook["cells"]
            if c["cell_type"] == "code" and source_tag in c.get("metadata", {}).get("tags", []))
        sources[name] = hashlib.sha256(source.encode()).hexdigest()
    return dict(artifact_role="method_demo", producer_notebook=producer.relative_to(root).as_posix(),
        method_source_sha256=sources[producer.stem], dependency_method_source_sha256=sources,
        dependency_source_sha256={name: hashlib.sha256((root / path).read_bytes()).hexdigest()
            for name, path in {"notebook_loader": "04_Research/a00_notebook_loader.py",
                               "data_contracts": "config/data_contracts.py"}.items()},
        upstream_demo_identity=upstream, parameters=parameters, scope=scope, timezone="Asia/Shanghai",
        runtime=dict(python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__,
                     pyarrow=pa.__version__, platform=platform.platform()))


def _save_real_demo(frame, path, schema, identity, sort_keys):
    """明确局部 Arrow Schema；同身份同内容保留原文件；安装后复读核对。"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    table = pa.Table.from_pandas(frame.loc[:, schema.names], schema=schema, preserve_index=False,
                               safe=True).replace_schema_metadata(schema.metadata).sort_by(sort_keys).combine_chunks()
    for field in schema:
        if not field.nullable and table[field.name].null_count:
            raise ValueError(f"demo 必需字段 {field.name} 为空")
    content = arrow_content_sha256(table, schema, sort_keys)
    expected = {**identity, "output_rows": table.num_rows, "output_sort_keys": sort_keys,
                "output_schema_sha256": hashlib.sha256(schema.serialize().to_pybytes()).hexdigest(),
                "output_content_sha256": content}
    expected = json.loads(json.dumps(expected, ensure_ascii=False, allow_nan=False))
    if path.exists():
        # 生产阶段检查旧文件自身完整性；当前版本由新计算结果的身份声明。
        old = _read_real_demo(path, require_current_sources=False)
        if {k: v for k, v in old[1].items() if k != "generated_at_utc"} == expected:
            return old[1]
    installed = {**expected, "generated_at_utc": datetime.now(timezone.utc).isoformat()}
    metadata = {**(schema.metadata or {}), b"demo_identity": json.dumps(installed, ensure_ascii=False,
        allow_nan=False, sort_keys=True).encode()}
    staged = path.with_name(path.name + ".staging")
    if staged.exists():
        raise RuntimeError(f"存在未处理 staging：{staged}")
    pq.write_table(table.replace_schema_metadata(metadata), staged, compression="zstd")
    _read_real_demo(staged, schema)
    os.replace(staged, path)
    _, actual = _read_real_demo(path, schema)
    assert actual == installed
    return installed


def _read_real_demo(path, schema=None, *, require_current_sources=True):
    path = Path(path).resolve()
    table = pq.read_table(path).combine_chunks()
    identity = json.loads(table.schema.metadata[b"demo_identity"])
    for name, expected_hash in (identity["dependency_method_source_sha256"].items() if require_current_sources else ()):
        source_path = (path.parent.parent.parent / "b03_Sampling/c01_SamplingPositions.ipynb"
            if name == "sampling_positions" else path.parent.parent / (name + ".ipynb"))
        notebook = json.loads(source_path.read_text(encoding="utf8"))
        source_tag = "export" if name == "sampling_positions" else "demo-dependency"
        source = "\n\n".join("".join(c["source"]) for c in notebook["cells"]
            if c["cell_type"] == "code" and source_tag in c.get("metadata", {}).get("tags", []))
        if hashlib.sha256(source.encode()).hexdigest() != expected_hash:
            raise RuntimeError(f"上游 {name} 计算定义已变化，请重算对应 demo")
    root = path.parents[4]
    for name, relative in {"notebook_loader": "04_Research/a00_notebook_loader.py",
                           "data_contracts": "config/data_contracts.py"}.items():
        if require_current_sources and hashlib.sha256((root / relative).read_bytes()).hexdigest() != identity["dependency_source_sha256"][name]:
            raise RuntimeError(f"demo 依赖 {name} 已变化，请重算")
    base_schema = schema
    if base_schema is None:
        encoded_schema = (pq.ParquetFile(path).metadata.metadata or {}).get(b"ARROW:schema")
        if encoded_schema is None:
            raise RuntimeError(f"缺少原始 Arrow Schema：{path}")
        base_schema = pa.ipc.read_schema(pa.BufferReader(base64.b64decode(encoded_schema))).remove_metadata()
    if not table.schema.equals(base_schema, check_metadata=False):
        raise RuntimeError(f"局部 Schema 不匹配：{path}")
    # 各局部 demo Schema 使用空 metadata，身份单独嵌入。
    if hashlib.sha256(base_schema.serialize().to_pybytes()).hexdigest() != identity["output_schema_sha256"]:
        raise RuntimeError(f"局部 Schema 身份不匹配：{path}")
    if table.num_rows != identity["output_rows"] or arrow_content_sha256(
        table, base_schema, identity["output_sort_keys"]) != identity["output_content_sha256"]:
        raise RuntimeError(f"demo 实际内容不匹配：{path}")
    return table.to_pandas(), identity


In [9]:
def _load_intraday_demo_inputs(root, *, start, end, test_start, calendar_path=None):
    """只读真实复权产物与正式日历；云端可传冻结的 silver 日历投影。"""
    candidate_root = Path(root)
    demo_base = candidate_root / "04_Research/a01_Methods"
    demo_variety = "RB"
    demo_source_range = "train20100101-20241231_test20250101-20260930"
    demo_start_date, demo_end_date, demo_test_start = map(pd.Timestamp, (start, end, test_start))
    import pyarrow.dataset as ds
    from config.data_contracts import FUTURES_BAR_CALENDAR_SCHEMA, arrow_to_pandas

    demo_adjusted_path = demo_base / "b02_MainContinuousAdjustment/c01_MainContinuousAdjustment_DemoData" / (
        f"c01_MainContinuousAdjustment_{demo_variety}_{demo_source_range}_ratio1.10_log_common_demo.parquet")
    demo_adjusted_identity = json.loads(pq.read_schema(demo_adjusted_path).metadata[b"demo_identity"])
    assert demo_adjusted_identity["parameters"]["test_start_date"] == "2025-01-01"
    assert demo_adjusted_identity["underlying_code"] == demo_variety
    demo_scope = {
        "minute_read_trading_dates": {"start": demo_start_date.date().isoformat(), "end": demo_end_date.date().isoformat()},
        "training_trading_dates": {"start": demo_start_date.date().isoformat(), "end": (demo_test_start - pd.Timedelta(days=1)).date().isoformat()},
        "test_trading_dates": {"start": demo_test_start.date().isoformat(), "end": demo_end_date.date().isoformat()},
        "interval_closed": "both", "timezone": "Asia/Shanghai",
    }
    demo_adjusted_columns = ["trading_date", "bar_at", "main_contract_code", "adjusted_log_close"]
    demo_main = ds.dataset(demo_adjusted_path, format="parquet").to_table(
        filter=((ds.field("is_main") == True)
            & (ds.field("trading_date") >= demo_start_date.date())
            & (ds.field("trading_date") <= demo_end_date.date())),
        columns=demo_adjusted_columns).to_pandas()
    demo_main = demo_main.sort_values("bar_at").reset_index(drop=True)
    demo_scope["actual_available_trading_dates"] = {
        "start": pd.Timestamp(demo_main["trading_date"].min()).date().isoformat(),
        "end": pd.Timestamp(demo_main["trading_date"].max()).date().isoformat(),
    }

    # 显式选择实际主力映射；日历包含没有行情的日期和真实夜盘变化。
    demo_mapping_path = demo_base / "b01_MainContractSelection/c01_MainContractSelection_DemoData" / (
        f"c01_MainContractSelection_{demo_variety}_20100101-20260930_ratio1.10_demo.parquet")
    demo_mapping = pq.read_table(demo_mapping_path, columns=["trading_date", "main_contract_code"]).to_pandas()
    demo_mapping["trading_date"] = pd.to_datetime(demo_mapping["trading_date"])
    demo_mapping = demo_mapping.loc[demo_mapping["trading_date"].between(demo_start_date, demo_end_date)].copy()
    calendar_columns = ["contract_code", "trading_date", "session_start_at", "session_end_at", "is_night_session"]
    calendar_schema = pa.schema([FUTURES_BAR_CALENDAR_SCHEMA.field(name) for name in calendar_columns])
    calendar_partitions = pa.schema([FUTURES_BAR_CALENDAR_SCHEMA.field(name) for name in
        FUTURES_BAR_CALENDAR_SCHEMA.metadata[b"partition_columns"].decode().split(",")])
    if calendar_path is None:
        from config.settings import settings
        calendar_dataset = ds.dataset(settings.futures_lake_root / "silver" /
            FUTURES_BAR_CALENDAR_SCHEMA.metadata[b"table_name"].decode(), schema=FUTURES_BAR_CALENDAR_SCHEMA,
            format="parquet", partitioning=ds.partitioning(calendar_partitions, flavor="hive"))
        calendar_filter = ((ds.field("bar_frequency") == "1m") & (ds.field("exchange_code") == "XSGE")
            & (ds.field("underlying_code") == demo_variety) & (ds.field("year") >= demo_start_date.year)
            & (ds.field("year") <= demo_end_date.year) & (ds.field("trading_date") >= demo_start_date.date())
            & (ds.field("trading_date") <= demo_end_date.date()))
        calendar_table = calendar_dataset.to_table(filter=calendar_filter, columns=calendar_columns)
    else:
        # 云输入是正式 silver 日历的字段投影，冻结时按原 Schema 保存。
        calendar_table = pq.read_table(calendar_path, columns=calendar_columns)
        selected = pd.to_datetime(calendar_table['trading_date'].to_pandas()).between(demo_start_date, demo_end_date)
        calendar_table = calendar_table.filter(pa.array(selected.to_numpy()))
    calendar_df = arrow_to_pandas(calendar_table, calendar_schema)
    calendar_df["trading_date"] = pd.to_datetime(calendar_df["trading_date"])
    demo_calendar = calendar_df.merge(demo_mapping, on="trading_date", how="inner")
    demo_calendar = demo_calendar.loc[demo_calendar["contract_code"].eq(demo_calendar["main_contract_code"]),
        ["trading_date", "session_start_at", "session_end_at", "is_night_session"]].sort_values(
        ["trading_date", "session_start_at"]).reset_index(drop=True)
    demo_calendar_schema = pa.schema([FUTURES_BAR_CALENDAR_SCHEMA.field(name) for name in demo_calendar.columns])
    calendar_selected_table = pa.Table.from_pandas(demo_calendar, schema=demo_calendar_schema, preserve_index=False)
    demo_calendar_hash = arrow_content_sha256(calendar_selected_table, demo_calendar_schema,
        [("trading_date", "ascending"), ("session_start_at", "ascending")])
    print(f"真实上游：{demo_variety}；{len(demo_main):,} 个主力分钟；{len(demo_calendar):,} 个 Session", flush=True)
    return dict(main=demo_main, calendar=demo_calendar, scope=demo_scope,
        upstream={"adjustment": demo_adjusted_identity,
                  "main_calendar": {"content_sha256": demo_calendar_hash, "rows": len(demo_calendar)}})


def _compute_intraday_demo(demo_main, demo_calendar, *, start, test_start,
                           measures=("rv", "bpv"), lookbacks=None):
    """同一套真实贡献、因果季节与回填验收；只返回内存结果，不保存文件。"""
    demo_start_date, demo_test_start = pd.Timestamp(start), pd.Timestamp(test_start)
    demo_lookbacks = {"1y": pd.DateOffset(years=1), "3y": pd.DateOffset(years=3)} if lookbacks is None else lookbacks
    demo_window_names = tuple(demo_lookbacks)
    demo_seasonality, curve_diagnostics = {}, {}
    demo_contributions = compute_minute_contributions(demo_main)
    demo_log_prices = demo_main["adjusted_log_close"].to_numpy(dtype=float, na_value=np.nan)
    demo_returns = demo_contributions["log_return"].to_numpy(dtype=float, na_value=np.nan)
    np.testing.assert_allclose(demo_returns[1:], np.diff(demo_log_prices), equal_nan=True)
    np.testing.assert_allclose(demo_contributions["rv_contribution"], demo_returns ** 2, equal_nan=True)
    np.testing.assert_allclose(demo_contributions["bpv_contribution"].iloc[1:],
        np.pi / 2 * np.abs(demo_returns[1:] * demo_returns[:-1]), equal_nan=True)
    demo_contribution_schema = pa.schema([
        pa.field("trading_date", pa.date32(), nullable=False),
        pa.field("bar_at", pa.timestamp("us", tz="Asia/Shanghai"), nullable=False),
        pa.field("main_contract_code", pa.string(), nullable=False),
        *[pa.field(name, pa.float64()) for name in
          ("adjusted_log_close", "log_return", "rv_contribution", "bpv_contribution")],
    ])
    demo_curve_schema = pa.schema([
        pa.field("trading_date", pa.date32(), nullable=False), pa.field("minute_slot", pa.time64("us"), nullable=False),
        pa.field("causal_factor", pa.float64()), pa.field("seasonal_factor", pa.float64()),
        pa.field("session_phase", pa.int64(), nullable=False), pa.field("history_start", pa.date32()),
        pa.field("history_end", pa.date32()), pa.field("sample_count", pa.int64(), nullable=False),
        pa.field("is_initial_backfill", pa.bool_(), nullable=False), pa.field("huber_location", pa.float64()),
        pa.field("status", pa.string(), nullable=False), pa.field("reason", pa.string(), nullable=False),
    ])
    for measure in measures:
        print(f"估计 {measure} 每日曲线：{tuple(demo_lookbacks)}", flush=True)
        causal = estimate_intraday_seasonality(demo_contributions, demo_calendar,
            contribution_col=f"{measure}_contribution", lookbacks=demo_lookbacks)
        filled = backfill_initial_seasonality(causal, demo_contributions, demo_calendar,
            train_start=demo_start_date, test_start=demo_test_start)
        demo_seasonality[measure] = filled
        assert filled["diagnostics"].loc[filled["diagnostics"]["is_initial_backfill"], "history_end"].dropna().lt(demo_test_start).all()
        for window_name in demo_window_names:
            curve = filled["curves"][window_name]
            pd.testing.assert_frame_equal(curve.loc[demo_test_start:], causal["curves"][window_name].loc[demo_test_start:])
            averages = curve.mean(axis=1).dropna()
            np.testing.assert_allclose(averages, 1.0, rtol=1e-10, atol=1e-10)
            diagnostics = filled["diagnostics"].loc[filled["diagnostics"]["window_name"].eq(window_name)].copy()
            diag_days = pd.DatetimeIndex(diagnostics["trading_date"])
            diag_slots = curve.columns.get_indexer(diagnostics["minute_slot"])
            diag_rows = curve.index.get_indexer(diag_days)
            diagnostics["seasonal_factor"] = curve.to_numpy()[diag_rows, diag_slots]
            diagnostics["causal_factor"] = causal["curves"][window_name].to_numpy()[diag_rows, diag_slots]
            curve_diagnostics[(measure, window_name)] = diagnostics
    return dict(contributions=demo_contributions, seasonality=demo_seasonality,
                curve_diagnostics=curve_diagnostics, contribution_schema=demo_contribution_schema,
                curve_schema=demo_curve_schema)


## 真实行情内存试算与完整云端 demo

内存试算的示例历史窗口为 30 点，完整云 demo 为 500 点。下面使用 RB 的 2024-12-31—2025-01-02 真实复权分钟行情，在 Notebook 内试算全部函数，仅展示结果，不写数据文件。完整 demo 使用上游 2010-01-01—2026-09-30 的全部可用序列，由云端调用下列同一套组合函数；每个目标点均用此前最多 500 个采样观测重新拟合，失败位置与原因保留。季节窗口 1 年／3 年、N=15、GK sigma×全合约成交额轴等是方法 demo 示例参数，正式大实验配置另行定义。


In [10]:
demo_real_inputs = _load_intraday_demo_inputs(candidate_root, start=demo_start_date,
    end=demo_end_date, test_start=demo_test_start)


真实上游：RB；570 个主力分钟；8 个 Session


In [11]:
_b04_intraday_preview = _compute_intraday_demo(demo_real_inputs["main"], demo_real_inputs["calendar"],
    start=demo_start_date, test_start=demo_test_start)
demo_contributions = _b04_intraday_preview["contributions"]
display(demo_contributions.head().rename(columns={
    "trading_date": "trading_date（交易日）", "bar_at": "bar_at（分钟末端时刻）",
    "main_contract_code": "main_contract_code（主力合约）", "adjusted_log_close": "adjusted_log_close（复权收盘 log 价格）",
    "log_return": "log_return（分钟收益）", "rv_contribution": "rv_contribution（平方收益贡献）",
    "bpv_contribution": "bpv_contribution（双幂变差贡献）"}).rename_axis("row（源分钟行）"))
for measure, filled in _b04_intraday_preview["seasonality"].items():
    display(filled["diagnostics"].groupby(["window_name", "status"]).size().rename("count（分钟位置数）").rename_axis(
        ["window_name（季节窗口）", "status（估计状态）"]).to_frame())
print("c01 真实试算完成：贡献、季节估计和初始回填；未写数据文件", flush=True)


估计 rv 每日曲线：('1y', '3y')


估计 bpv 每日曲线：('1y', '3y')


,trading_date（交易日）,bar_at（分钟末端时刻）,main_contract_code（主力合约）,adjusted_log_close（复权收盘 log 价格）,log_return（分钟收益）,rv_contribution（平方收益贡献）,bpv_contribution（双幂变差贡献）
row（源分钟行）,,,,,,,
0,2024-12-31,2024-12-30 21:01:00+08:00,RB2505.XSGE,8.107813,NaN,NaN,NaN
1,2024-12-31,2024-12-30 21:02:00+08:00,RB2505.XSGE,8.107512,-3.015216e-04,9.091525e-08,NaN
2,2024-12-31,2024-12-30 21:03:00+08:00,RB2505.XSGE,8.107512,-2.713631e-07,7.363793e-14,1.285254e-10
3,2024-12-31,2024-12-30 21:04:00+08:00,RB2505.XSGE,8.106909,-6.030441e-04,3.636622e-07,2.570513e-10
4,2024-12-31,2024-12-30 21:05:00+08:00,RB2505.XSGE,8.107210,3.011604e-04,9.069761e-08,2.852771e-07


count（分钟位置数）
window_name（季节窗口） status（估计状态）                  
1y                no_valid_history             2
                  ok                         688
3y                no_valid_history             2
                  ok                         688

count（分钟位置数）
window_name（季节窗口） status（估计状态）                  
1y                no_valid_history             4
                  ok                         686
3y                no_valid_history             4
                  ok                         686

c01 真实试算完成：贡献、季节估计和初始回填；未写数据文件
